In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

def chat(text):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=[{"role": "user", "content": text}],
    )
    # Keep only the text pieces, skipping any "thinking" piece
    return "".join(block.text for block in message.content if block.type == "text")

print(chat("Define quantum computing in one sentence"))

In [ ]:
import json

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=[], output_format=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if output_format:
        params["output_config"] = {"format": output_format}

    response = client.messages.create(**params)
    # Keep only the text pieces, skipping any "thinking" piece
    return "".join(block.text for block in response.content if block.type == "text")

In [ ]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" or "json" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects for each format (3 python, 3 json, 3 regex), 9 objects in total.
"""

    # The course uses prefilling ("```json") + a stop sequence to get clean JSON.
    # Sonnet 5.5 doesn't allow prefilling, so we ask for JSON with a fixed shape instead.
    output_format = {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "tasks": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "task": {"type": "string"},
                            "format": {"type": "string", "enum": ["python", "json", "regex"]},
                        },
                        "required": ["task", "format"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["tasks"],
            "additionalProperties": False,
        },
    }

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, output_format=output_format)
    return json.loads(text)["tasks"]

In [ ]:
dataset = generate_dataset()
print(dataset)

In [ ]:
import os

# Keep the golden examples from earlier runs, if dataset.json already has some
golden_examples = {}
if os.path.exists("dataset.json"):
    try:
        with open("dataset.json", "r") as f:
            old_data = json.load(f)
        if isinstance(old_data, dict):
            golden_examples = old_data.get("golden_examples", {})
    except json.JSONDecodeError:
        pass  # empty or broken file: start without golden examples

# Save the dataset so it can be loaded later during evaluation
data = {"tasks": dataset, "golden_examples": golden_examples}
with open("dataset.json", "w") as f:
    json.dump(data, f, indent=2)

In [ ]:
def strip_code_fences(text):
    """Removes a surrounding ```...``` block, if Claude added one"""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else ""
        if text.rstrip().endswith("```"):
            text = text.rstrip()[:-3]
    return text.strip()

def bullet_list(items):
    return "\n".join(f"- {item}" for item in items)

def run_prompt(test_case, example=None, failures=None):
    """Merges the prompt and test case input, then returns the result"""
    format = test_case["format"]

    prompt = f"""
Please solve the following task:

<task>
{test_case["task"]}
</task>

<instructions>
* Respond only with {format}
* Do not add any comments or commentary or explanation
* Do not wrap your answer in markdown code fences
</instructions>
"""

    # Show Claude the best answer so far for this format, and why it's ideal
    if example:
        prompt += f"""
Here is an example input with an ideal response for a similar {format} task:

<example>
<example_task>
{example["task"]}
</example_task>
<ideal_response>
{example["output"]}
</ideal_response>
<why_ideal>
{bullet_list(example.get("why_ideal", []))}
</why_ideal>
</example>

Match the style and quality of the ideal response, but solve the task in <task>, not the example task.
"""

    # Warn Claude about the mistakes earlier answers made for this format
    if failures:
        prompt += f"""
<common_failures>
Previous {format} answers lost points for these issues. Avoid them:
{bullet_list(failures)}
</common_failures>
"""
    
    messages = []
    add_user_message(messages, prompt)
    # The course prefills "```code" to get raw code; Sonnet 5.5 doesn't allow
    # prefilling, so we ask for no fences above and strip any that slip through
    output = strip_code_fences(chat(messages))
    return output

In [ ]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    # The course code never fills in {task} and {solution}; .format() does that
    eval_prompt = eval_prompt.format(task=test_case["task"], solution=output)

    # The course uses prefilling ("```json") + a stop sequence to get clean JSON.
    # Sonnet 5.5 doesn't allow prefilling, so we ask for JSON with a fixed shape instead.
    output_format = {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "strengths": {"type": "array", "items": {"type": "string"}},
                "weaknesses": {"type": "array", "items": {"type": "string"}},
                "reasoning": {"type": "string"},
                "score": {"type": "integer"},
            },
            "required": ["strengths", "weaknesses", "reasoning", "score"],
            "additionalProperties": False,
        },
    }

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages, output_format=output_format)
    return json.loads(eval_text)

In [ ]:
def grade_by_resemblance(test_case, output, example):
    """Scores how closely the output follows the golden example"""
    eval_prompt = f"""
You are an expert code reviewer. Compare an AI-generated solution against a reference example.
The two solve different tasks, so do not judge the task content. Judge how closely the solution
follows the example's style: structure, naming, conventions, level of detail, and formatting.

Example task: {example["task"]}
Example answer:
{example["output"]}

Task: {test_case["task"]}
Solution:
{output}

Provide your evaluation as a structured JSON object with:
- "reasoning": A concise explanation of how closely the solution resembles the example
- "score": A number between 1-10, where 10 means it follows the example's style exactly
"""

    output_format = {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "reasoning": {"type": "string"},
                "score": {"type": "integer"},
            },
            "required": ["reasoning", "score"],
            "additionalProperties": False,
        },
    }

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages, output_format=output_format)
    return json.loads(eval_text)

In [ ]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    """Picks the validator that matches the test case's format"""
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
from statistics import mean

def run_test_case(test_case, example=None, failures=None):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case, example, failures)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    scores = [model_score, syntax_score]

    # Only scored when there is a golden example for this format
    resemblance_score = None
    if example:
        resemblance_grade = grade_by_resemblance(test_case, output, example)
        resemblance_score = resemblance_grade["score"]
        scores.append(resemblance_score)

    score = mean(scores)
    
    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "resemblance_score": resemblance_score,
        "score": score,
        "reasoning": reasoning,
        "strengths": model_grade["strengths"],
        "weaknesses": model_grade["weaknesses"]
    }

In [ ]:
def save_data(data):
    with open("dataset.json", "w") as f:
        json.dump(data, f, indent=2)

def run_eval(data):
    """Runs every test case, then keeps the best answer per format as its golden example"""
    tasks = data["tasks"]
    golden_examples = data.setdefault("golden_examples", {})
    common_failures = data.setdefault("common_failures", {})

    # Golden examples saved before "why_ideal" existed: ask the grader why they're good
    for format, golden in golden_examples.items():
        if "why_ideal" not in golden:
            grade = grade_by_model({"task": golden["task"], "format": format}, golden["output"])
            golden["why_ideal"] = grade["strengths"]
            save_data(data)

    # This run uses the golden examples and failures as they were when it started;
    # new ones found during the run are saved for the next run
    examples = dict(golden_examples)
    failures_to_avoid = dict(common_failures)

    results = []
    
    for test_case in tasks:
        format = test_case["format"]

        example = examples.get(format)
        # Don't show a task its own answer as the example
        if example and example["task"] == test_case["task"]:
            example = None

        result = run_test_case(test_case, example, failures_to_avoid.get(format))
        results.append(result)

        # Show the grades and their average for this response
        resemblance = result["resemblance_score"] if result["resemblance_score"] is not None else "-"
        print(f"[{format}] {test_case['task']}")
        print(f"  Model score: {result['model_score']}  |  Syntax score: {result['syntax_score']}  |  Resemblance score: {resemblance}  |  Average: {result['score']:.2f}")

        # A higher average than the current golden example replaces it
        best = golden_examples.get(format)
        if best is None or result["score"] > best["score"]:
            golden_examples[format] = {
                "task": test_case["task"],
                "output": result["output"],
                "score": result["score"],
                "why_ideal": result["strengths"],
            }
            save_data(data)
            print(f"  -> New golden example for {format}")
        print()

    # The weaknesses of each format's lowest-scoring answer become the failures to avoid next run
    for format in {r["test_case"]["format"] for r in results}:
        worst = min((r for r in results if r["test_case"]["format"] == format), key=lambda r: r["score"])
        common_failures[format] = worst["weaknesses"]
    save_data(data)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score:.2f}")
    
    return results

In [ ]:
with open("dataset.json", "r") as f:
    data = json.load(f)

results = run_eval(data)

In [ ]:
print(json.dumps(results, indent=2))